# ⚙️ Análisis Exploratorio de Datos (EDA) - Mantenimiento Predictivo de Rodamientos (CWRU)

**Curso:** Data Science con Python – 2026-I  
**Docente:** Alexander Quispe  
**Estudiante (Founder):** John Barraza  
**Startup:** MineAssist-PdM  

Este notebook desarrolla el Análisis Exploratorio de Datos (EDA) y la validación matemática del modelo de clasificación de fallas de rodamientos a partir del **Case Western Reserve University (CWRU) Bearing Dataset**. Las fallas de rodamientos representan una de las principales causas de paradas mecánicas no programadas en equipos mineros de alta criticidad como molinos de bolas, chancadoras y fajas transportadoras.

---

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

## 1. Carga del Dataset y Estructura

Cargamos el dataset reducido procesado con las características estadísticas extraídas de las señales de vibración de aceleración de CWRU.

In [ ]:
# Ruta al dataset relativo al proyecto
data_path = "../data/cwru_bearing_small.csv"
df = pd.read_csv(data_path)

# Vista inicial
print(f"Dimensiones del dataset: {df.shape}")
df.head()

Las variables del dataset corresponden a:
- `rms`: Valor eficaz (Root Mean Square) de la señal de aceleración. Indica la energía general de vibración.
- `kurtosis`: Cuarto momento estadístico. Mide la presencia de picos/impactos severos (típicos de grietas y descascarillados).
- `crest_factor`: Relación pico/RMS de la vibración.
- `skewness`: Asimetría de la señal.
- `temperature`: Temperatura operativa registrada en el rodamiento.
- `speed`: Velocidad del eje (RPM).
- `fault`: Condición del rodamiento (Target: `Normal`, `Inner_Race`, `Outer_Race`, `Ball`).
- `fault_diameter`: Diámetro del defecto físico inducido artificialmente (en pulgadas).

## 2. Análisis Descriptivo e Inspectivo

In [ ]:
# Resumen estadístico
df.describe().T

In [ ]:
# Distribución de clases
plt.figure(figsize=(6, 4))
sns.countplot(x='fault', data=df, palette='viridis')
plt.title('Distribución de Clases de Fallas de Rodamientos')
plt.xlabel('Condición')
plt.ylabel('Cantidad de Muestras')
plt.show()

El dataset cuenta con un balance perfecto entre las 4 categorías para la demo (10 muestras de cada una).

## 3. Visualización de Correlaciones y Distribución por Falla

In [ ]:
# Boxplots para analizar cómo cada variable diferencia las fallas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x='fault', y='rms', data=df, ax=axes[0,0], palette='Set2')
axes[0,0].set_title('Energía de Vibración (RMS) vs. Tipo de Falla')

sns.boxplot(x='fault', y='kurtosis', data=df, ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Factor de Curtosis (Severidad de Impacto) vs. Tipo de Falla')

sns.boxplot(x='fault', y='temperature', data=df, ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Temperatura vs. Tipo de Falla')

sns.boxplot(x='fault', y='crest_factor', data=df, ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Factor de Cresta vs. Tipo de Falla')

plt.tight_layout()
plt.show()

### Observaciones Clave:
1. **RMS:** El valor RMS distingue perfectamente la clase `Normal` (cercana a 0.08 g) de las fallas. Las fallas en la pista externa (`Outer_Race`) presentan la mayor energía de vibración en promedio.
2. **Curtosis:** La curtosis aumenta drásticamente en rodamientos dañados, alcanzando su valor máximo en `Outer_Race` (>7.0), indicando impactos repetitivos muy fuertes.
3. **Temperatura:** Existe una correlación directa entre la gravedad de la falla mecánica y el incremento térmico de los rodamientos.

## 4. Entrenamiento y Evaluación del Modelo Clasificador

In [ ]:
# Codificación del Target
encoder = LabelEncoder()
df['fault_encoded'] = encoder.fit_transform(df['fault'])

# Variables predictoras y variable objetivo
features = ['rms', 'kurtosis', 'crest_factor', 'skewness', 'temperature', 'speed']
X = df[features]
y = df['fault_encoded']

# División del dataset (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Entrenamiento del Clasificador Random Forest
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Predicciones y Reporte de Métricas
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión General del Modelo (Accuracy): {accuracy * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

In [ ]:
# Importancia de Características (Feature Importance)
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 4))
sns.barplot(x=[features[i] for i in indices], y=importances[indices], palette='mako')
plt.title('Importancia de Características en la Predicción de Fallas')
plt.ylabel('Peso relativo')
plt.xlabel('Features')
plt.show()

### Conclusiones del Modelamiento:
* El clasificador demuestra una precisión óptima en la separación de fallas a partir de features de vibración.
* Las variables **RMS** y **Curtosis** son las más influyentes para el diagnóstico de rodamientos de apoyo mecánicos en fajas y molinos.